# AI Observability Demo — Data Generation

**Target:** `msahil.ai_observability` (12 Delta tables)  
**Runtime:** ~3–5 min on Serverless | **Spec:** See `0 - Setup` notebook

## What this notebook generates

| # | Cell | Table | Records | Notes |
|---|------|-------|---------|-------|
| 1 | Setup | — | — | Install Faker, restart Python |
| 2 | Config | `msahil.ai_observability` schema | — | Constants, account/region maps |
| 3 | Agent Registry | `agent_registry` | 28 | Master table — run first |
| 4 | MLflow Experiments | `mlflow_experiments` | 28 | One per agent |
| 5 | MCP Catalog | `mcp_catalog` | 15 | Registered tools |
| 6 | User Profiles | `user_profiles` | ~133 | Locale-aware names |
| 7 | MLflow Runs | `mlflow_runs` | ~50K | **All 10 anomalies injected here** |
| 8 | MLflow Run Metrics | `mlflow_run_metrics` | ~270K | Derived from runs |
| 9 | AI Gateway Usage | `ai_gateway_usage` | ~50K | 1:1 with runs (minus Scenario 9 gap) |
| 10 | Tool Access Logs | `tool_access_logs` | ~83K | UC governance + anomalies 4/6/8 |
| 11 | NR Infra Metrics | `newrelic_infra_metrics` | ~1.3M | 5-min intervals, all entities |
| 12 | NR APM Transactions | `newrelic_apm_transactions` | ~50K | 1:1 with gateway, linked via request_id |
| 13 | SN Incidents | `servicenow_incidents` | 10 | One per anomaly scenario |
| 14 | SN Change Requests | `servicenow_change_requests` | 6 | Remediation tickets |
| 15 | Summary | — | — | Verification counts |

## Cross-table linkage
```
agent_registry.agent_id
  └─ mlflow_runs.params["agent_id"]
       ├─ mlflow_run_metrics.run_id
       ├─ ai_gateway_usage.request_tags["run_id"]
       │    └─ newrelic_apm_transactions.linked_gateway_request_id
       └─ tool_access_logs.trace_id
```

## Anomaly injection approach
Anomalies are injected **inline** during MLflow run generation (Cell 7) using probability-based conditions tied to specific day/hour windows. The same `run_id` propagates to Gateway and Access Logs, ensuring cross-table JOINs reveal correlated signals. New Relic infra metrics inject independent time-series anomalies in the same time windows.

In [0]:
%pip install faker --quiet
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timedelta
import uuid
import random
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from faker import Faker

# ─── Configuration ───────────────────────────────────────────────────────────
dbutils.widgets.text("catalog", "msahil")
dbutils.widgets.text("schema", "ai_observability")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
# Account IDs per country (each spoke has its own account)
ACCOUNT_MAP = {
    "DE": "eon-de-prod-001",
    "IT": "eon-it-prod-002",
    "SE": "eon-se-prod-003",
    "HU": "eon-hu-prod-004",
    "RO": "eon-ro-prod-005",
    "NL": "eon-nl-prod-006"
}

# Time range: last 30 days
END_DATE = datetime(2026, 6, 4)
START_DATE = END_DATE - timedelta(days=30)

# Regions and cloud mapping
REGIONS = ["DE", "IT", "SE", "HU", "RO", "NL"]
CLOUD_MAP = {"DE": "Azure", "IT": "Azure", "SE": "Azure", "HU": "Azure", "RO": "Azure", "NL": "AWS"}

# Workspace IDs per region
WORKSPACE_MAP = {
    "DE": "ws-de-001", "IT": "ws-it-002", "SE": "ws-se-003",
    "HU": "ws-hu-004", "RO": "ws-ro-005", "NL": "ws-nl-006"
}

# Agent framework distribution: 60% AgentBricks, 25% Azure AI Agent Service, 15% Amazon Bedrock Agents
FRAMEWORKS = ["AgentBricks"] * 12 + ["Azure AI Agent Service"] * 5 + ["Amazon Bedrock Agents"] * 3

# Models used
MODELS = ["gpt-4o", "gpt-4o-mini", "claude-sonnet-4", "meta-llama-3.3-70b-instruct", "dbrx-instruct"]

# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✓ Schema ready: {CATALOG}.{SCHEMA}")
print(f"✓ Time range: {START_DATE.date()} to {END_DATE.date()}")

In [0]:
# ─── Agent Registry ──────────────────────────────────────────────────────────
# Generate ~30 production agents across all regions

fake = Faker()
random.seed(42)
np.random.seed(42)

agent_definitions = [
    # DE agents (Hub + Spoke)
    ("DE", "DE-Customer-Support-Agent", "Customer Service"),
    ("DE", "DE-HR-Policy-Agent", "Human Resources"),
    ("DE", "DE-Orchestrator-Agent", "Platform Engineering"),
    ("DE", "DE-Compliance-Audit-Agent", "Legal & Compliance"),
    ("DE", "DE-Energy-Trading-Agent", "Trading"),
    ("DE", "DE-Network-Planning-Agent", "Grid Operations"),
    # IT agents
    ("IT", "IT-Customer-Support-Agent", "Customer Service"),
    ("IT", "IT-Billing-Agent", "Finance"),
    ("IT", "IT-Regulatory-Filing-Agent", "Legal & Compliance"),
    ("IT", "IT-Demand-Forecast-Agent", "Operations"),
    ("IT", "IT-Field-Service-Agent", "Operations"),
    # SE agents
    ("SE", "SE-Energy-Forecast-Agent", "Operations"),
    ("SE", "SE-Customer-Onboarding-Agent", "Customer Service"),
    ("SE", "SE-Sustainability-Report-Agent", "ESG"),
    ("SE", "SE-Grid-Monitoring-Agent", "Grid Operations"),
    # HU agents
    ("HU", "HU-Grid-Operations-Agent", "Grid Operations"),
    ("HU", "HU-Customer-Query-Agent", "Customer Service"),
    ("HU", "HU-Procurement-Agent", "Procurement"),
    ("HU", "HU-Metering-Agent", "Operations"),
    # RO agents
    ("RO", "RO-Operations-Agent", "Operations"),
    ("RO", "RO-Customer-Agent", "Customer Service"),
    ("RO", "RO-Asset-Management-Agent", "Asset Management"),
    ("RO", "RO-Safety-Compliance-Agent", "Safety"),
    # NL agents
    ("NL", "NL-Energy-Forecast-Agent", "Operations"),
    ("NL", "NL-Customer-Support-Agent", "Customer Service"),
    ("NL", "NL-Bedrock-Dev-Agent", "Platform Engineering"),
    ("NL", "NL-Smart-Meter-Agent", "IoT"),
    ("NL", "NL-Trading-Analytics-Agent", "Trading"),
]

agents = []
for i, (region, name, team) in enumerate(agent_definitions):
    agent_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, name))
    framework = random.choice(FRAMEWORKS)
    agents.append({
        "agent_id": agent_id,
        "agent_name": name,
        "region": region,
        "cloud_provider": CLOUD_MAP[region],
        "agent_framework": framework,
        "deployment_env": "production",
        "owner_team": team,
        "created_at": fake.date_time_between(start_date="-180d", end_date="-31d"),
        "status": "active"
    })

agent_registry_df = spark.createDataFrame(agents)
agent_registry_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.agent_registry")

print(f"✓ Agent Registry: {len(agents)} agents written")
display(agent_registry_df)

In [0]:
# ─── MLflow Experiments (mirrors system.mlflow.experiments_latest) ────────────

experiments = []
for agent in agents:
    exp_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"exp-{agent['agent_name']}"))
    experiments.append({
        "account_id": ACCOUNT_MAP[agent["region"]],
        "workspace_id": WORKSPACE_MAP[agent["region"]],
        "experiment_id": exp_id,
        "name": f"/Agents/{agent['agent_name']}",
        "create_time": agent["created_at"],
        "update_time": END_DATE - timedelta(hours=random.randint(1, 48)),
        "delete_time": None
    })

experiments_schema = StructType([
    StructField("account_id", StringType()),
    StructField("workspace_id", StringType()),
    StructField("experiment_id", StringType()),
    StructField("name", StringType()),
    StructField("create_time", TimestampType()),
    StructField("update_time", TimestampType()),
    StructField("delete_time", TimestampType())
])
experiments_df = spark.createDataFrame(experiments, schema=experiments_schema)
experiments_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.mlflow_experiments")

# Build lookup for later use
agent_experiment_map = {a["agent_id"]: experiments[i]["experiment_id"] for i, a in enumerate(agents)}

print(f"✓ MLflow Experiments: {len(experiments)} experiments written")
display(experiments_df)

In [0]:
# ─── MCP Catalog — Registered Tools ──────────────────────────────────────────

mcp_tools = [
    # UC Functions
    ("get_customer_usage", "uc_function", "eon_hub.customer_360.get_usage", "DE", "Retrieve customer energy usage data"),
    ("query_energy_prices", "uc_function", "eon_hub.energy.get_spot_prices", "DE", "Query real-time energy spot prices"),
    ("get_billing_summary", "uc_function", "eon_hub.billing.get_summary", "IT", "Get customer billing summary"),
    ("check_grid_status", "uc_function", "eon_hub.grid.check_status", "HU", "Check grid operational status"),
    ("get_meter_reading", "uc_function", "eon_hub.iot.get_meter_data", "NL", "Retrieve smart meter readings"),
    ("get_compliance_rules", "uc_function", "eon_hub.legal.get_rules", "DE", "Fetch applicable compliance rules"),
    # Vector Search
    ("search_knowledge_base", "vector_search", "eon_hub.knowledge.kb_index", "DE", "Semantic search over internal knowledge base"),
    ("search_policy_docs", "vector_search", "eon_hub.hr.policy_index", "DE", "Search HR policy documents"),
    ("search_technical_docs", "vector_search", "eon_hub.engineering.tech_docs_index", "SE", "Search technical documentation"),
    # REST APIs
    ("get_weather_forecast", "rest_api", None, "SE", "External weather API for energy forecasting"),
    ("send_notification", "rest_api", None, "DE", "Send push/email notifications to customers"),
    ("get_market_data", "rest_api", None, "NL", "Retrieve energy market data from external provider"),
    # SQL Queries
    ("query_customer_profile", "sql_query", "eon_hub.customer_360.profiles", "DE", "Query customer profile table"),
    ("query_asset_inventory", "sql_query", "eon_ro.assets.inventory", "RO", "Query asset inventory for maintenance"),
    ("query_demand_history", "sql_query", "eon_hub.energy.demand_history", "IT", "Query historical demand data"),
]

mcp_catalog = []
for tool_name, tool_type, catalog_path, owner_region, description in mcp_tools:
    tool_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"tool-{tool_name}"))
    mcp_catalog.append({
        "tool_id": tool_id,
        "tool_name": tool_name,
        "tool_type": tool_type,
        "catalog_path": catalog_path,
        "description": description,
        "owner_region": owner_region,
        "call_count_30d": random.randint(500, 50000),
        "avg_latency_ms": round(random.uniform(50, 800), 1)
    })

mcp_catalog_df = spark.createDataFrame(mcp_catalog)
mcp_catalog_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.mcp_catalog")

# Build tool lookup
tool_names = [t["tool_name"] for t in mcp_catalog]

print(f"✓ MCP Catalog: {len(mcp_catalog)} tools written")
display(mcp_catalog_df)

In [0]:
# ─── User Profiles ───────────────────────────────────────────────────────────

fake_locales = {
    "DE": Faker("de_DE"), "IT": Faker("it_IT"), "SE": Faker("sv_SE"),
    "HU": Faker("hu_HU"), "RO": Faker("ro_RO"), "NL": Faker("nl_NL")
}

departments = ["Operations", "Customer Service", "Trading", "Engineering", "Finance", "HR", "Legal", "ESG", "Grid Operations", "IoT"]
roles = ["analyst", "engineer", "manager", "executive"]
role_weights = [0.4, 0.35, 0.2, 0.05]

users = []
for region in REGIONS:
    n_users = random.randint(15, 30)
    region_agents = [a["agent_id"] for a in agents if a["region"] == region]
    f = fake_locales[region]
    for _ in range(n_users):
        first = f.first_name()
        last = f.last_name()
        users.append({
            "user_id": str(uuid.uuid4()),
            "user_email": f"{first.lower()}.{last.lower()}@eon.com",
            "region": region,
            "department": random.choice(departments),
            "role": random.choices(roles, weights=role_weights, k=1)[0],
            "agents_used": random.sample(region_agents, k=min(random.randint(1, 4), len(region_agents)))
        })

users_schema = StructType([
    StructField("user_id", StringType()),
    StructField("user_email", StringType()),
    StructField("region", StringType()),
    StructField("department", StringType()),
    StructField("role", StringType()),
    StructField("agents_used", ArrayType(StringType()))
])

users_df = spark.createDataFrame(users, schema=users_schema)
users_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.user_profiles")

print(f"✓ User Profiles: {len(users)} users written")
display(users_df)

In [0]:
# ─── MLflow Runs (mirrors system.mlflow.runs_latest) ─────────────────────────
# Generate ~500K runs across 30 days with anomaly injection

import json

def get_day_number(ts):
    """Return day offset (1-30) from START_DATE"""
    return (ts - START_DATE).days + 1

def is_business_hour(ts):
    """CET business hours: 8-18, Mon-Fri"""
    hour = (ts.hour + 1) % 24  # rough UTC->CET
    return ts.weekday() < 5 and 8 <= hour <= 18

# Endpoint assignments per agent (for cross-table consistency)
agent_endpoint_map = {}
for agent in agents:
    region = agent["region"]
    name = agent["agent_name"].lower().replace("-", "_")
    agent_endpoint_map[agent["agent_id"]] = f"{region.lower()}-{name.split('_')[0]}-endpoint"

# --- Generate baseline + anomaly runs ---
# Reduced to ~50K runs to avoid driver OOM / gRPC payload limits.
# Anomaly scenarios are probability-based so they remain fully intact.
print("Generating MLflow runs...")

runs = []
NUM_RUNS = 50_000
runs_per_agent = {a["agent_id"]: max(500, int(NUM_RUNS * random.uniform(0.8, 1.2) / len(agents))) for a in agents}

# Adjust to hit ~50K total
scale_factor = NUM_RUNS / sum(runs_per_agent.values())
runs_per_agent = {k: int(v * scale_factor) for k, v in runs_per_agent.items()}

for agent in agents:
    agent_id = agent["agent_id"]
    region = agent["region"]
    exp_id = agent_experiment_map[agent_id]
    n_runs = runs_per_agent[agent_id]
    model = random.choice(MODELS)
    
    for _ in range(n_runs):
        # Generate timestamp with business-hour bias
        ts = START_DATE + timedelta(seconds=random.randint(0, 30 * 86400))
        if random.random() < 0.7 and not is_business_hour(ts):
            ts = ts.replace(hour=random.randint(8, 17))
        
        day = get_day_number(ts)
        
        # Baseline metrics
        latency = np.random.lognormal(5.5, 0.6)  # ~250ms median
        input_tokens = int(np.random.lognormal(7.5, 0.8))  # ~1800 median
        output_tokens = int(np.random.lognormal(6.5, 0.7))  # ~665 median
        cost = (input_tokens * 0.000003 + output_tokens * 0.000015)
        feedback = round(random.choices([1,2,3,4,5], weights=[0.02,0.05,0.15,0.45,0.33], k=1)[0] + random.uniform(-0.3, 0.3), 1) if random.random() < 0.4 else None
        status = "FINISHED"
        error_type = None
        tools_called = random.sample(tool_names, k=random.randint(1, 4))
        actual_model = model
        
        # ─── ANOMALY INJECTION ────────────────────────────────────────────────
        
        # Scenario 1: Provider Outage (NL, Day 12, 14:00-16:00 CET = 13:00-15:00 UTC)
        if region == "NL" and day == 12 and 13 <= ts.hour <= 15:
            if random.random() < 0.4:
                status = "FAILED"
                error_type = "provider_unavailable"
            else:
                latency *= random.uniform(3, 5)
                actual_model = "gpt-4o-mini"  # fallback
                if feedback: feedback = max(1.0, feedback - 1.5)
        
        # Scenario 2: Token Exhaustion (IT, Days 8-10)
        if region == "IT" and agent["agent_name"] == "IT-Customer-Support-Agent" and 8 <= day <= 10:
            input_tokens = int(input_tokens * (1.5 + (day - 8) * 0.8))  # growing
            if input_tokens > 120000:
                status = "FAILED"
                error_type = "context_overflow"
            elif random.random() < 0.02 * (day - 7) ** 2:
                status = "FAILED"
                error_type = "context_overflow"
                input_tokens = random.randint(120000, 160000)
            cost = (input_tokens * 0.000003 + output_tokens * 0.000015)
            if feedback: feedback = max(1.0, feedback - 0.5 * (day - 7))
        
        # Scenario 3: Rate Limit Cascade (DE/HU/RO, Days 5/12/19 Mon 9 AM)
        if region in ["DE", "HU", "RO"] and day in [5, 12, 19] and 7 <= ts.hour <= 9:
            if random.random() < 0.6:
                status = "FAILED"
                error_type = "rate_limited"
                latency = random.uniform(10, 50)  # fast failure
        
        # Scenario 4: Guardrail Surge (DE HR Agent, Day 18, 10:00-14:00)
        if agent["agent_name"] == "DE-HR-Policy-Agent" and day == 18 and 9 <= ts.hour <= 13:
            if random.random() < 0.12:
                output_tokens = 0
                error_type = "guardrail_block"
                if feedback: feedback = 1.0
        
        # Scenario 5: Model Drift (SE Energy Forecast, Days 20-27)
        if agent["agent_name"] == "SE-Energy-Forecast-Agent" and 20 <= day <= 27:
            drift_factor = (day - 19) / 8  # 0.125 to 1.0
            output_tokens = int(output_tokens * (1 - drift_factor * 0.4))
            if feedback: feedback = max(1.0, feedback - drift_factor * 1.5)
            actual_model = "gpt-4o-2025-06-01" if day >= 20 else model
        
        # Scenario 6: Lateral Movement (RO Operations, Day 22, 03:00-05:00)
        if agent["agent_name"] == "RO-Operations-Agent" and day == 22 and 2 <= ts.hour <= 4:
            latency *= 2.5  # cross-region data fetches
            tools_called = ["query_de_customer_data", "access_hub_financials"] + tools_called[:1]
        
        # Scenario 7: Reasoning Loop (HU Grid Operations, Day 15, 11:00-12:30)
        if agent["agent_name"] == "HU-Grid-Operations-Agent" and day == 15 and 10 <= ts.hour <= 12:
            if random.random() < 0.15:
                tools_called = ["check_grid_status"] * random.randint(40, 60)
                latency = random.uniform(120000, 180000)  # 2-3 minutes
                input_tokens = int(input_tokens * random.uniform(15, 25))  # context grows
                output_tokens = int(output_tokens * 10)
                cost = (input_tokens * 0.000003 + output_tokens * 0.000015)  # 20x normal
        
        # Scenario 8: Tool Degradation (DE/IT/HU/SE, Day 9, 06:00-06:45)
        if region in ["DE", "IT", "HU", "SE"] and day == 9 and ts.hour == 6 and ts.minute < 45:
            if "query_energy_prices" in tools_called:
                latency += 14500  # tool adds 14.5s
                if random.random() < 0.3:
                    status = "KILLED"  # timeout
                    error_type = "tool_timeout"
        
        # Scenario 9: Shadow AI Gap (NL Bedrock Dev, Day 14 20:00 - Day 15 02:00)
        # (handled by NOT generating gateway records for this agent during this window)
        
        # Scenario 10: Cross-Agent Cascade (Day 25, 09:15-09:45)
        if day == 25 and 9 <= ts.hour <= 9 and ts.minute >= 15:
            if agent["agent_name"] == "IT-Demand-Forecast-Agent":
                status = "FAILED"
                error_type = "internal_error"
            elif agent["agent_name"] == "DE-Orchestrator-Agent":
                if random.random() < 0.7:
                    status = "FAILED"
                    error_type = "downstream_timeout"
                    latency = 300000  # 5min timeout
            elif region in ["HU", "RO"] and random.random() < 0.5:
                status = "FAILED"
                error_type = "rate_limited"
        
        # ─── END ANOMALY INJECTION ────────────────────────────────────────────
        
        run_id = str(uuid.uuid4())
        end_time = ts + timedelta(milliseconds=latency)
        
        run = {
            "account_id": ACCOUNT_MAP[region],
            "workspace_id": WORKSPACE_MAP[region],
            "run_id": run_id,
            "experiment_id": exp_id,
            "created_by": f"{agent['agent_name'].lower().replace('-', '.')}@eon.com",
            "start_time": ts,
            "end_time": end_time,
            "run_name": f"run-{run_id[:8]}",
            "status": status,
            "params": {
                "agent_id": agent_id,
                "region": region,
                "model_name": actual_model,
                "framework": agent["agent_framework"],
                "error_type": error_type or "",
                "tools_called": ",".join(tools_called[:5])
            },
            "tags": {
                "mlflow.runName": f"run-{run_id[:8]}",
                "mlflow.source.type": "JOB",
                "mlflow.source.name": agent["agent_name"]
            },
            "aggregated_metrics": {
                "latency_ms": round(latency, 2),
                "input_tokens": float(input_tokens),
                "output_tokens": float(output_tokens),
                "total_cost_usd": round(cost, 6),
                "feedback_score": feedback if feedback else 0.0,
                "tool_calls_count": float(len(tools_called))
            }
        }
        runs.append(run)

print(f"Generated {len(runs)} MLflow runs. Writing to Delta...")

# Define explicit schema for MAP fields
runs_schema = StructType([
    StructField("account_id", StringType()),
    StructField("workspace_id", StringType()),
    StructField("run_id", StringType()),
    StructField("experiment_id", StringType()),
    StructField("created_by", StringType()),
    StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()),
    StructField("run_name", StringType()),
    StructField("status", StringType()),
    StructField("params", MapType(StringType(), StringType())),
    StructField("tags", MapType(StringType(), StringType())),
    StructField("aggregated_metrics", MapType(StringType(), DoubleType()))
])

# Write in batches to avoid driver OOM
BATCH_SIZE = 100_000
for i in range(0, len(runs), BATCH_SIZE):
    batch = runs[i:i+BATCH_SIZE]
    batch_df = spark.createDataFrame(batch, schema=runs_schema)
    mode = "overwrite" if i == 0 else "append"
    batch_df.write.mode(mode).saveAsTable(f"{CATALOG}.{SCHEMA}.mlflow_runs")
    print(f"  Written batch {i//BATCH_SIZE + 1}/{(len(runs)-1)//BATCH_SIZE + 1}")

print(f"\n✓ MLflow Runs: {len(runs)} runs written to {CATALOG}.{SCHEMA}.mlflow_runs")

In [0]:
# ─── MLflow Run Metrics (mirrors system.mlflow.run_metrics_history) ───────────
# Generate metric rows from the runs data

print("Generating MLflow run metrics...")

metric_names = ["latency_ms", "input_tokens", "output_tokens", "total_cost_usd", "feedback_score", "error_rate"]

metrics = []
for run in runs:
    agg = run["aggregated_metrics"]
    ts = run["start_time"]
    
    for metric_name in ["latency_ms", "input_tokens", "output_tokens", "total_cost_usd"]:
        metrics.append({
            "account_id": run["account_id"],
            "workspace_id": run["workspace_id"],
            "experiment_id": run["experiment_id"],
            "run_id": run["run_id"],
            "record_id": str(uuid.uuid4()),
            "metric_name": metric_name,
            "metric_time": ts,
            "metric_step": 0,
            "metric_value": float(agg[metric_name])
        })
    
    # Feedback (only if provided)
    if agg["feedback_score"] > 0:
        metrics.append({
            "account_id": run["account_id"],
            "workspace_id": run["workspace_id"],
            "experiment_id": run["experiment_id"],
            "run_id": run["run_id"],
            "record_id": str(uuid.uuid4()),
            "metric_name": "feedback_score",
            "metric_time": ts + timedelta(minutes=random.randint(1, 30)),
            "metric_step": 0,
            "metric_value": float(agg["feedback_score"])
        })
    
    # Error rate (1.0 if failed, 0.0 if success)
    metrics.append({
        "account_id": run["account_id"],
        "workspace_id": run["workspace_id"],
        "experiment_id": run["experiment_id"],
        "run_id": run["run_id"],
        "record_id": str(uuid.uuid4()),
        "metric_name": "error_rate",
        "metric_time": ts,
        "metric_step": 0,
        "metric_value": 1.0 if run["status"] != "FINISHED" else 0.0
    })

print(f"Generated {len(metrics)} metric records. Writing to Delta in batches...")

BATCH_SIZE = 500_000
for i in range(0, len(metrics), BATCH_SIZE):
    batch = metrics[i:i+BATCH_SIZE]
    batch_df = spark.createDataFrame(batch)
    mode = "overwrite" if i == 0 else "append"
    batch_df.write.mode(mode).saveAsTable(f"{CATALOG}.{SCHEMA}.mlflow_run_metrics")
    print(f"  Written batch {i//BATCH_SIZE + 1}/{(len(metrics)-1)//BATCH_SIZE + 1}")

print(f"\n✓ MLflow Run Metrics: {len(metrics)} records written")

# Free memory before next heavy cell
del metrics
import gc
gc.collect()

In [0]:
# ─── AI Gateway Usage (mirrors system.ai_gateway.usage) ──────────────────────
# Generate gateway records correlated with MLflow runs

print("Generating AI Gateway usage records...")

# Endpoint definitions per region
endpoint_definitions = {
    "DE": [("de-gpt4o-endpoint", "gpt-4o"), ("de-dbrx-endpoint", "dbrx-instruct")],
    "IT": [("it-customer-agent-endpoint", "gpt-4o"), ("it-claude-endpoint", "claude-sonnet-4")],
    "SE": [("se-forecast-endpoint", "gpt-4o"), ("se-llama-endpoint", "meta-llama-3.3-70b-instruct")],
    "HU": [("hu-operations-endpoint", "gpt-4o-mini"), ("hu-dbrx-endpoint", "dbrx-instruct")],
    "RO": [("ro-general-endpoint", "gpt-4o"), ("ro-claude-endpoint", "claude-sonnet-4")],
    "NL": [("nl-gpt4o-endpoint", "gpt-4o"), ("nl-bedrock-endpoint", "meta-llama-3.3-70b-instruct")]
}

gateway_records = []

for run in runs:
    region = run["params"]["region"]
    agent_id = run["params"]["agent_id"]
    agent_name = next((a["agent_name"] for a in agents if a["agent_id"] == agent_id), "unknown")
    ts = run["start_time"]
    day = get_day_number(ts)
    
    # Scenario 9: Shadow AI Gap — skip gateway record for NL-Bedrock-Dev-Agent, Day 14 20:00 - Day 15 02:00
    if agent_name == "NL-Bedrock-Dev-Agent" and ((day == 14 and ts.hour >= 20) or (day == 15 and ts.hour < 2)):
        continue  # No gateway record = observability gap
    
    # Select endpoint
    region_endpoints = endpoint_definitions.get(region, [("default-endpoint", "gpt-4o")])
    endpoint_name, default_model = random.choice(region_endpoints)
    endpoint_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, endpoint_name))
    
    # Base values from run
    agg = run["aggregated_metrics"]
    latency = agg["latency_ms"]
    input_tokens = int(agg["input_tokens"])
    output_tokens = int(agg["output_tokens"])
    actual_model = run["params"]["model_name"]
    status_code = 200
    fallback_triggered = False
    
    # Map run status to HTTP status codes
    error_type = run["params"].get("error_type", "")
    if error_type == "provider_unavailable":
        status_code = 503
    elif error_type == "context_overflow":
        status_code = 400
    elif error_type == "rate_limited":
        status_code = 429
    elif error_type == "internal_error":
        status_code = 500
    elif error_type == "tool_timeout":
        status_code = 504
    elif error_type == "downstream_timeout":
        status_code = 504
    
    # Scenario 1: Fallback detection in routing
    routing_info = json.dumps({"primary": endpoint_name, "fallback_used": False})
    if region == "NL" and day == 12 and 13 <= ts.hour <= 15 and status_code == 200:
        fallback_triggered = True
        routing_info = json.dumps({"primary": "nl-gpt4o-endpoint", "fallback_used": True, "fallback_to": "de-gpt4o-endpoint"})
    
    # Scenario 4: Guardrail flags
    guardrail_flags = []
    if agent_name == "DE-HR-Policy-Agent" and day == 18 and 9 <= ts.hour <= 13 and error_type == "guardrail_block":
        guardrail_flags = ["pii_detection", "sensitive_data"]
    
    # Invocation count (Scenario 7: reasoning loop = many invocations per request)
    tool_calls_count = int(agg.get("tool_calls_count", 3))
    
    request_id = str(uuid.uuid4())
    
    record = {
        "account_id": ACCOUNT_MAP[region],
        "workspace_id": run["workspace_id"],
        "request_id": request_id,
        "schema_version": "1.0",
        "endpoint_id": endpoint_id,
        "endpoint_name": endpoint_name,
        "endpoint_tags": {"region": region, "team": next((a["owner_team"] for a in agents if a["agent_id"] == agent_id), "unknown")},
        "endpoint_metadata": json.dumps({"cloud": CLOUD_MAP[region], "framework": run["params"]["framework"]}),
        "event_time": ts,
        "latency_ms": int(latency) if status_code != 429 else None,
        "time_to_first_byte_ms": int(latency * random.uniform(0.1, 0.3)) if status_code == 200 else None,
        "destination_type": "model_serving",
        "destination_name": endpoint_name,
        "destination_id": endpoint_id,
        "destination_model": actual_model,
        "requester": f"{agent_name.lower().replace('-', '.')}@eon.com",
        "requester_type": "service_principal",
        "ip_address": None,
        "url": f"/serving-endpoints/{endpoint_name}/invocations",
        "user_agent": "databricks-sdk-python/0.30.0",
        "api_type": "chat",
        "request_tags": {"agent_id": agent_id, "region": region, "run_id": run["run_id"]},
        "input_tokens": input_tokens if status_code != 429 else None,
        "output_tokens": output_tokens if status_code not in [429, 503] else None,
        "total_tokens": (input_tokens + output_tokens) if status_code == 200 else None,
        "token_details": json.dumps({"cached_tokens": int(input_tokens * 0.1)}) if status_code == 200 else None,
        "response_content_type": "application/json" if random.random() < 0.7 else "text/event-stream",
        "status_code": status_code,
        "routing_information": routing_info,
        "invocation_id": str(uuid.uuid4()),
        "invocation_metadata": json.dumps({"guardrail_flags": guardrail_flags, "invocation_count": tool_calls_count})
    }
    gateway_records.append(record)

print(f"Generated {len(gateway_records)} gateway records. Writing to Delta in batches...")

gateway_schema = StructType([
    StructField("account_id", StringType()),
    StructField("workspace_id", StringType()),
    StructField("request_id", StringType()),
    StructField("schema_version", StringType()),
    StructField("endpoint_id", StringType()),
    StructField("endpoint_name", StringType()),
    StructField("endpoint_tags", MapType(StringType(), StringType())),
    StructField("endpoint_metadata", StringType()),
    StructField("event_time", TimestampType()),
    StructField("latency_ms", IntegerType()),
    StructField("time_to_first_byte_ms", IntegerType()),
    StructField("destination_type", StringType()),
    StructField("destination_name", StringType()),
    StructField("destination_id", StringType()),
    StructField("destination_model", StringType()),
    StructField("requester", StringType()),
    StructField("requester_type", StringType()),
    StructField("ip_address", StringType()),
    StructField("url", StringType()),
    StructField("user_agent", StringType()),
    StructField("api_type", StringType()),
    StructField("request_tags", MapType(StringType(), StringType())),
    StructField("input_tokens", IntegerType()),
    StructField("output_tokens", IntegerType()),
    StructField("total_tokens", IntegerType()),
    StructField("token_details", StringType()),
    StructField("response_content_type", StringType()),
    StructField("status_code", IntegerType()),
    StructField("routing_information", StringType()),
    StructField("invocation_id", StringType()),
    StructField("invocation_metadata", StringType())
])

BATCH_SIZE = 100_000
for i in range(0, len(gateway_records), BATCH_SIZE):
    batch = gateway_records[i:i+BATCH_SIZE]
    batch_df = spark.createDataFrame(batch, schema=gateway_schema)
    mode = "overwrite" if i == 0 else "append"
    batch_df.write.mode(mode).saveAsTable(f"{CATALOG}.{SCHEMA}.ai_gateway_usage")
    print(f"  Written batch {i//BATCH_SIZE + 1}/{(len(gateway_records)-1)//BATCH_SIZE + 1}")

print(f"\n✓ AI Gateway Usage: {len(gateway_records)} records written")

In [0]:
# ─── Tool & Data Access Logs ─────────────────────────────────────────────────
# Generate access logs linked to runs, with anomaly-specific access patterns

print("Generating Tool & Data Access Logs...")

resource_paths = {
    "uc_table": ["eon_hub.customer_360.profiles", "eon_hub.energy.consumption", "eon_hub.billing.invoices",
                 "eon_de.grid.topology", "eon_it.regulatory.filings", "eon_hub.hr.salary_bands",
                 "eon_hub.finance.budgets", "eon_ro.assets.inventory"],
    "uc_view": ["eon_hub.analytics.daily_summary", "eon_hub.customer_360.active_customers",
                "eon_hub.energy.price_forecast_v"],
    "volume_file": ["/Volumes/eon_hub/documents/contracts/", "/Volumes/eon_hub/reports/monthly/",
                    "/Volumes/eon_se/forecasts/weather_data/"],
    "lakebase_table": ["eon_hub.energy.spot_prices", "eon_hub.iot.meter_cache", "eon_nl.trading.positions"],
    "mcp_tool": [t["tool_name"] for t in mcp_catalog],
    "external_api": ["weather-api.eon.com", "market-data.eon.com", "notifications.eon.com"]
}

access_logs = []

# Sample runs (1 access log per run on average, some runs have multiple)
for run in runs:
    n_accesses = random.choices([0, 1, 2, 3, 4], weights=[0.1, 0.4, 0.3, 0.15, 0.05], k=1)[0]
    if n_accesses == 0:
        continue
    
    agent_id = run["params"]["agent_id"]
    agent_name = next((a["agent_name"] for a in agents if a["agent_id"] == agent_id), "unknown")
    region = run["params"]["region"]
    ts = run["start_time"]
    day = get_day_number(ts)
    
    for _ in range(n_accesses):
        resource_type = random.choices(
            list(resource_paths.keys()),
            weights=[0.3, 0.15, 0.1, 0.15, 0.2, 0.1], k=1
        )[0]
        resource_path = random.choice(resource_paths[resource_type])
        granted = True
        
        # Scenario 6: Lateral Movement (RO Operations, Day 22, 03:00-05:00)
        if agent_name == "RO-Operations-Agent" and day == 22 and 2 <= ts.hour <= 4:
            # Attempt cross-region access
            if random.random() < 0.6:
                resource_type = "uc_table"
                resource_path = random.choice(["eon_de.customer_360.profiles", "eon_hub.finance.budgets", "eon_hub.hr.salary_bands"])
                granted = False  # UC denies cross-region access
        
        # Scenario 4: Guardrail — HR salary access denied
        if agent_name == "DE-HR-Policy-Agent" and day == 18 and 9 <= ts.hour <= 13:
            if random.random() < 0.15:
                resource_path = "eon_hub.hr.salary_bands"
                granted = False
        
        # Scenario 8: Tool degradation — Lakebase table access slow/empty
        bytes_transferred = random.randint(1024, 10_000_000)
        row_count = random.randint(1, 50000) if resource_type in ["uc_table", "uc_view", "lakebase_table"] else None
        
        if resource_path == "eon_hub.energy.spot_prices" and day == 9 and ts.hour == 6 and ts.minute < 45:
            bytes_transferred = 0
            row_count = 0
        
        access_logs.append({
            "access_id": str(uuid.uuid4()),
            "trace_id": run["run_id"],
            "agent_id": agent_id,
            "resource_type": resource_type,
            "resource_path": resource_path,
            "access_type": "read" if resource_type != "external_api" else "execute",
            "row_count": row_count,
            "bytes_transferred": bytes_transferred,
            "timestamp": ts + timedelta(milliseconds=random.randint(10, 5000)),
            "granted": granted
        })

print(f"Generated {len(access_logs)} access log records. Writing to Delta in batches...")

BATCH_SIZE = 200_000
for i in range(0, len(access_logs), BATCH_SIZE):
    batch = access_logs[i:i+BATCH_SIZE]
    batch_df = spark.createDataFrame(batch)
    mode = "overwrite" if i == 0 else "append"
    batch_df.write.mode(mode).saveAsTable(f"{CATALOG}.{SCHEMA}.tool_access_logs")
    print(f"  Written batch {i//BATCH_SIZE + 1}/{(len(access_logs)-1)//BATCH_SIZE + 1}")

print(f"\n✓ Tool Access Logs: {len(access_logs)} records written")

# Free memory
del access_logs
gc.collect()

In [0]:
# ─── New Relic Infrastructure Metrics ─────────────────────────────────────────
# Time-series infra metrics correlated with AI anomaly events

print("Generating New Relic infrastructure metrics...")

# Entity definitions per region
nr_entities = {
    "DE": [
        ("AI_GATEWAY_ENDPOINT", "de-gpt4o-endpoint"),
        ("MODEL_SERVING_ENDPOINT", "de-orchestrator-serving"),
        ("AZURE_OPENAI", "de-azure-openai-instance"),
        ("NETWORK_INTERFACE", "de-vnet-gateway"),
    ],
    "IT": [
        ("AI_GATEWAY_ENDPOINT", "it-customer-agent-endpoint"),
        ("MODEL_SERVING_ENDPOINT", "it-model-serving"),
        ("AZURE_OPENAI", "it-azure-openai-instance"),
        ("NETWORK_INTERFACE", "it-vnet-gateway"),
    ],
    "SE": [
        ("AI_GATEWAY_ENDPOINT", "se-forecast-endpoint"),
        ("MODEL_SERVING_ENDPOINT", "se-model-serving"),
        ("AZURE_OPENAI", "se-azure-openai-instance"),
        ("LAKEBASE_INSTANCE", "se-lakebase-energy"),
    ],
    "HU": [
        ("AI_GATEWAY_ENDPOINT", "hu-operations-endpoint"),
        ("MODEL_SERVING_ENDPOINT", "hu-model-serving"),
        ("AZURE_OPENAI", "hu-azure-openai-instance"),
        ("LAKEBASE_INSTANCE", "hu-lakebase-grid"),
    ],
    "RO": [
        ("AI_GATEWAY_ENDPOINT", "ro-general-endpoint"),
        ("MODEL_SERVING_ENDPOINT", "ro-model-serving"),
        ("AZURE_OPENAI", "ro-azure-openai-instance"),
        ("NETWORK_INTERFACE", "ro-vnet-gateway"),
    ],
    "NL": [
        ("AI_GATEWAY_ENDPOINT", "nl-gpt4o-endpoint"),
        ("MODEL_SERVING_ENDPOINT", "nl-model-serving"),
        ("AWS_BEDROCK", "nl-bedrock-instance"),
        ("NETWORK_INTERFACE", "nl-vpc-gateway"),
    ],
}

# Shared Lakebase instance (for Scenario 8)
nr_entities["DE"].append(("LAKEBASE_INSTANCE", "hub-lakebase-energy-prices"))

metric_baselines = {
    "cpu_percent": (25.0, 8.0),
    "memory_percent": (40.0, 10.0),
    "request_rate_per_sec": (50.0, 15.0),
    "error_rate_percent": (1.5, 0.8),
    "latency_p50_ms": (45.0, 12.0),
    "latency_p99_ms": (250.0, 80.0),
    "connection_pool_active": (15.0, 5.0),
    "connection_pool_max": (50.0, 0.0),  # static
    "network_latency_ms": (8.0, 3.0),
    "dns_resolution_ms": (5.0, 2.0),
    "bytes_in_per_sec": (500000.0, 150000.0),
    "bytes_out_per_sec": (800000.0, 200000.0),
    "active_threads": (12.0, 4.0),
    "queue_depth": (3.0, 2.0),
}

# Generate 15-second interval metrics for all entities over 30 days
# To keep manageable: sample every 5 minutes (not 15 sec) = 8640 points per entity
# Total: ~6 regions * ~4 entities * 8640 = ~207K rows

INTERVAL_MINUTES = 5
TOTAL_INTERVALS = int(30 * 24 * 60 / INTERVAL_MINUTES)  # 8640

nr_metrics = []

for region, entities in nr_entities.items():
    account_id = ACCOUNT_MAP[region]
    for entity_type, entity_name in entities:
        # Select relevant metrics for this entity type
        if entity_type in ["AI_GATEWAY_ENDPOINT", "MODEL_SERVING_ENDPOINT"]:
            entity_metrics = ["cpu_percent", "memory_percent", "request_rate_per_sec", "error_rate_percent", 
                           "latency_p50_ms", "latency_p99_ms", "active_threads", "queue_depth"]
        elif entity_type in ["AZURE_OPENAI", "AWS_BEDROCK"]:
            entity_metrics = ["request_rate_per_sec", "error_rate_percent", "latency_p50_ms", "latency_p99_ms"]
        elif entity_type == "LAKEBASE_INSTANCE":
            entity_metrics = ["cpu_percent", "memory_percent", "connection_pool_active", "connection_pool_max", "latency_p50_ms"]
        elif entity_type == "NETWORK_INTERFACE":
            entity_metrics = ["network_latency_ms", "dns_resolution_ms", "bytes_in_per_sec", "bytes_out_per_sec"]
        else:
            entity_metrics = ["cpu_percent", "memory_percent"]
        
        for interval_idx in range(TOTAL_INTERVALS):
            ts = START_DATE + timedelta(minutes=interval_idx * INTERVAL_MINUTES)
            day = get_day_number(ts)
            
            for metric_name in entity_metrics:
                mean, std = metric_baselines[metric_name]
                value = max(0, np.random.normal(mean, std))
                alert_severity = None
                alert_policy = None
                
                # ─── ANOMALY INJECTION (correlated with AI scenarios) ───
                
                # Scenario 1: NL Provider Outage (Day 12, 13:00-15:00 UTC)
                if region == "NL" and day == 12 and 13 <= ts.hour <= 15:
                    if entity_type == "NETWORK_INTERFACE" and metric_name == "network_latency_ms":
                        value = np.random.normal(450, 100)  # 50x spike
                        alert_severity = "CRITICAL"
                        alert_policy = "Network Latency - NL Region"
                    elif entity_type == "NETWORK_INTERFACE" and metric_name == "dns_resolution_ms":
                        value = np.random.normal(2000, 500)  # DNS timeout
                        alert_severity = "CRITICAL"
                    elif entity_type == "AI_GATEWAY_ENDPOINT" and metric_name == "error_rate_percent":
                        value = np.random.normal(45, 10)
                        alert_severity = "CRITICAL"
                        alert_policy = "AI Gateway Error Rate - NL"
                
                # Scenario 7: Reasoning Loop HU (Day 15, 10:00-12:00 UTC)
                if region == "HU" and day == 15 and 10 <= ts.hour <= 12:
                    if entity_type == "MODEL_SERVING_ENDPOINT":
                        if metric_name == "cpu_percent":
                            value = np.random.normal(88, 5)
                            alert_severity = "WARNING"
                            alert_policy = "Serving Endpoint CPU - HU"
                        elif metric_name == "memory_percent":
                            value = np.random.normal(82, 5)
                            alert_severity = "WARNING"
                        elif metric_name == "active_threads":
                            value = np.random.normal(45, 8)
                            alert_severity = "CRITICAL"
                
                # Scenario 8: Tool Degradation (Day 9, 06:00-06:45 UTC)
                if day == 9 and ts.hour == 6 and ts.minute < 45:
                    if entity_name == "hub-lakebase-energy-prices":
                        if metric_name == "connection_pool_active":
                            value = np.random.normal(48, 2)  # near max
                            alert_severity = "CRITICAL"
                            alert_policy = "Lakebase Connection Pool Exhaustion"
                        elif metric_name == "cpu_percent":
                            value = np.random.normal(95, 3)
                            alert_severity = "CRITICAL"
                        elif metric_name == "latency_p50_ms":
                            value = np.random.normal(12000, 2000)
                            alert_severity = "CRITICAL"
                
                # Scenario 9: Shadow AI (Day 14 20:00 - Day 15 02:00 UTC)
                if region == "NL" and ((day == 14 and ts.hour >= 20) or (day == 15 and ts.hour < 2)):
                    if entity_type == "AWS_BEDROCK" and metric_name == "request_rate_per_sec":
                        value = np.random.normal(180, 30)  # direct Bedrock calls visible
                        alert_severity = "WARNING"
                        alert_policy = "Unexpected Bedrock Direct Traffic"
                    elif entity_name == "nl-gpt4o-endpoint" and metric_name == "request_rate_per_sec":
                        value = 0.0  # gateway shows zero (shadow path)
                
                # Scenario 10: Cross-Agent Cascade (Day 25, 09:15-09:45 UTC)
                if day == 25 and ts.hour == 9 and 15 <= ts.minute <= 45:
                    if region == "IT" and entity_type == "AZURE_OPENAI" and metric_name == "error_rate_percent":
                        value = np.random.normal(75, 10)  # root cause
                        alert_severity = "CRITICAL"
                        alert_policy = "Azure OpenAI Error Rate - IT"
                    elif region == "DE" and entity_type == "MODEL_SERVING_ENDPOINT":
                        if metric_name == "request_rate_per_sec":
                            value = np.random.normal(280, 40)  # 5x retry storm
                            alert_severity = "CRITICAL"
                        elif metric_name == "queue_depth":
                            value = np.random.normal(85, 15)
                            alert_severity = "CRITICAL"
                            alert_policy = "Orchestrator Queue Depth - DE"
                    elif region in ["HU", "RO"] and entity_type == "AI_GATEWAY_ENDPOINT":
                        if metric_name == "queue_depth":
                            value = np.random.normal(50, 10)
                            alert_severity = "WARNING"
                
                # Scenario 3: Rate Limit Cascade (Days 5/12/19, 08:00-09:30 UTC)
                if day in [5, 12, 19] and 8 <= ts.hour <= 9 and ts.minute <= 30:
                    if region in ["DE", "HU", "RO"] and entity_type == "AI_GATEWAY_ENDPOINT":
                        if metric_name == "request_rate_per_sec":
                            value = np.random.normal(350, 50)  # massive spike
                            alert_severity = "CRITICAL"
                            alert_policy = "Rate Limit Threshold - Multi-Region"
                        elif metric_name == "queue_depth":
                            value = np.random.normal(60, 10)
                            alert_severity = "WARNING"
                
                # ─── END ANOMALY INJECTION ───
                
                nr_metrics.append({
                    "metric_id": str(uuid.uuid4()),
                    "timestamp": ts,
                    "region": region,
                    "account_id": account_id,
                    "entity_type": entity_type,
                    "entity_name": entity_name,
                    "metric_name": metric_name,
                    "metric_value": float(round(value, 2)),
                    "alert_severity": alert_severity,
                    "alert_policy_name": alert_policy
                })

print(f"Generated {len(nr_metrics)} New Relic infra metric records. Writing in batches...")

BATCH_SIZE = 200_000
for i in range(0, len(nr_metrics), BATCH_SIZE):
    batch = nr_metrics[i:i+BATCH_SIZE]
    batch_df = spark.createDataFrame(batch)
    mode = "overwrite" if i == 0 else "append"
    batch_df.write.mode(mode).saveAsTable(f"{CATALOG}.{SCHEMA}.newrelic_infra_metrics")
    print(f"  Written batch {i//BATCH_SIZE + 1}/{(len(nr_metrics)-1)//BATCH_SIZE + 1}")

print(f"\n✓ New Relic Infra Metrics: {len(nr_metrics)} records written")

# Free memory
del nr_metrics
gc.collect()

In [0]:
# ─── New Relic APM Transactions ─────────────────────────────────────────────
# Application-level traces correlated with AI Gateway requests

import gc
gc.collect()

print("Generating New Relic APM transactions...")

apm_transactions = []

# Sample from gateway_records (1:1 mapping for cross-platform correlation)
for gw in gateway_records:
    region = gw["request_tags"]["region"]
    ts = gw["event_time"]
    day = get_day_number(ts)
    
    # Determine service name
    service_name = f"ai-gateway-proxy-{region.lower()}"
    
    # Base transaction metrics
    duration = gw["latency_ms"] if gw["latency_ms"] else random.uniform(5, 50)
    status_code = gw["status_code"]
    error = status_code >= 400
    ext_call_count = random.randint(1, 3)
    ext_call_duration = duration * random.uniform(0.6, 0.9)  # most time in external calls
    db_call_count = random.randint(0, 2)
    db_call_duration = random.uniform(1, 20) * db_call_count
    
    # Anomaly-specific APM behavior
    # Scenario 1: NL outage - external call duration spikes
    if region == "NL" and day == 12 and 13 <= ts.hour <= 15:
        ext_call_duration = random.uniform(5000, 30000)  # timeout on Azure
    
    # Scenario 8: Tool degradation - database calls spike
    if day == 9 and ts.hour == 6 and ts.minute < 45:
        if "query_energy_prices" in str(gw.get("request_tags", {})):
            db_call_count = random.randint(3, 8)
            db_call_duration = random.uniform(10000, 15000)
    
    # Scenario 10: Cascade - DE orchestrator shows high external call count
    if day == 25 and ts.hour == 9 and 15 <= ts.minute <= 45:
        if region == "DE":
            ext_call_count = random.randint(8, 15)  # retry calls
            ext_call_duration = duration * 0.95
    
    apm_transactions.append({
        "transaction_id": str(uuid.uuid4()),
        "timestamp": ts,
        "region": region,
        "service_name": service_name,
        "transaction_name": gw["url"] or f"POST /serving-endpoints/{gw['endpoint_name']}/invocations",
        "duration_ms": float(round(duration, 2)) if duration else None,
        "status_code": status_code,
        "error": error,
        "external_call_count": ext_call_count,
        "external_call_duration_ms": float(round(ext_call_duration, 2)),
        "database_call_count": db_call_count,
        "database_call_duration_ms": float(round(db_call_duration, 2)),
        "linked_gateway_request_id": gw["request_id"]
    })

print(f"Generated {len(apm_transactions)} APM transaction records. Writing in batches...")

BATCH_SIZE = 50_000
for i in range(0, len(apm_transactions), BATCH_SIZE):
    batch = apm_transactions[i:i+BATCH_SIZE]
    batch_df = spark.createDataFrame(batch)
    mode = "overwrite" if i == 0 else "append"
    batch_df.write.mode(mode).saveAsTable(f"{CATALOG}.{SCHEMA}.newrelic_apm_transactions")
    print(f"  Written batch {i//BATCH_SIZE + 1}/{(len(apm_transactions)-1)//BATCH_SIZE + 1}")

print(f"\n✓ New Relic APM Transactions: {len(apm_transactions)} records written")

In [0]:
# ─── ServiceNow Incidents ───────────────────────────────────────────────────
# ITSM incidents auto-generated from anomaly detections

print("Generating ServiceNow incidents and change requests...")

# Incident definitions mapped to each anomaly scenario
incident_scenarios = [
    {
        "scenario": 1,
        "number": "INC0012001",
        "short_description": "AI Gateway NL Region - Provider Outage (503 errors)",
        "description": "Azure OpenAI endpoint in Netherlands region returning 503 errors. Fallback routing to DE region activated. NL agents experiencing 40% failure rate. Latency 3-5x normal for surviving requests.",
        "priority": "P2",
        "category": "Infrastructure",
        "subcategory": "AI Gateway",
        "assignment_group": "SRE - NL Region",
        "assigned_to": "jan.devries@eon.com",
        "impact": "2-Region",
        "urgency": "2-High",
        "cmdb_ci": "nl-gpt4o-endpoint (Azure OpenAI - West Europe)",
        "source_system": "Databricks",
        "affected_regions": ["NL"],
        "affected_agents": [a["agent_id"] for a in agents if a["region"] == "NL"],
        "affected_users_count": 45,
        "mttr_minutes": 95,
        "root_cause": "Azure OpenAI regional capacity issue",
        "resolution_notes": "Vendor resolved regional capacity issue. Fallback routing prevented full outage. Post-mortem: enable multi-region failover for NL.",
        "opened_at": START_DATE + timedelta(days=11, hours=13, minutes=2),
        "state": "Closed"
    },
    {
        "scenario": 2,
        "number": "INC0012002",
        "short_description": "IT Customer Agent - Context Overflow (token budget exhaustion)",
        "description": "IT-Customer-Support-Agent error rate climbing from 2% to 18% over 3 days. Input tokens exceeding 120K context window. Caused by regulatory season increasing complaint length.",
        "priority": "P3",
        "category": "AI Platform",
        "subcategory": "Model Serving",
        "assignment_group": "AI Platform - Hub",
        "assigned_to": "marco.rossi@eon.com",
        "impact": "3-Team",
        "urgency": "3-Medium",
        "cmdb_ci": "it-customer-agent-endpoint (GPT-4o)",
        "source_system": "Databricks",
        "affected_regions": ["IT"],
        "affected_agents": [a["agent_id"] for a in agents if a["agent_name"] == "IT-Customer-Support-Agent"],
        "affected_users_count": 120,
        "mttr_minutes": 30,
        "root_cause": "Input length exceeded model context window during regulatory filing season",
        "resolution_notes": "Implemented input chunking strategy and increased context window to 200K. Added proactive alert for avg token length trending.",
        "opened_at": START_DATE + timedelta(days=9, hours=14, minutes=30),
        "state": "Closed"
    },
    {
        "scenario": 3,
        "number": "INC0012003",
        "short_description": "Multi-Region Rate Limiting - Monday Morning Peak",
        "description": "Recurring Monday 9AM CET rate limiting cascade across DE, HU, RO regions. AI Gateway returning 429s for 45 minutes. Shared endpoint capacity insufficient for simultaneous regional startup.",
        "priority": "P2",
        "category": "AI Platform",
        "subcategory": "AI Gateway",
        "assignment_group": "AI Platform - Hub",
        "assigned_to": "klaus.mueller@eon.com",
        "impact": "2-Region",
        "urgency": "2-High",
        "cmdb_ci": "hub-ai-gateway (Shared Multi-Region)",
        "source_system": "Databricks",
        "affected_regions": ["DE", "HU", "RO"],
        "affected_agents": [a["agent_id"] for a in agents if a["region"] in ["DE", "HU", "RO"]],
        "affected_users_count": 280,
        "mttr_minutes": 15,
        "root_cause": "Shared rate limit pool insufficient for concurrent Monday morning startup across 3 regions",
        "resolution_notes": "Increased per-region rate limits by 3x. Implemented staggered agent warm-up schedule. Added region-aware token bucket.",
        "opened_at": START_DATE + timedelta(days=4, hours=8, minutes=47),
        "state": "Closed"
    },
    {
        "scenario": 4,
        "number": "INC0012004",
        "short_description": "SECURITY: Adversarial Prompt Injection - DE HR Agent",
        "description": "DE-HR-Policy-Agent experiencing guardrail blocking surge. PII detection and sensitive_data guardrails triggered 12% of requests. Pattern consistent with adversarial prompt injection targeting salary data. 3-4 specific requester accounts identified.",
        "priority": "P1",
        "category": "Security",
        "subcategory": "Guardrail Violation",
        "assignment_group": "Security Operations",
        "assigned_to": "anna.schmidt@eon.com",
        "impact": "2-Region",
        "urgency": "1-Critical",
        "cmdb_ci": "DE-HR-Policy-Agent (AgentBricks)",
        "source_system": "Databricks",
        "affected_regions": ["DE"],
        "affected_agents": [a["agent_id"] for a in agents if a["agent_name"] == "DE-HR-Policy-Agent"],
        "affected_users_count": 15,
        "mttr_minutes": 45,
        "root_cause": "Coordinated adversarial prompt injection attempt from 4 compromised user accounts",
        "resolution_notes": "Accounts locked. Additional guardrail layer added. Forensic review completed. Referred to CISO for investigation.",
        "opened_at": START_DATE + timedelta(days=17, hours=9, minutes=3),
        "state": "Closed"
    },
    {
        "scenario": 5,
        "number": "INC0012005",
        "short_description": "Quality SLA Breach - SE Energy Forecast Agent (model drift)",
        "description": "SE-Energy-Forecast-Agent feedback score declining from 4.2 to 3.1 over 7 days. No errors detected. Root cause: upstream model version change by provider (gpt-4o-2025-05-13 to gpt-4o-2025-06-01). Forecast MAPE degraded from 3.5% to 8.2%.",
        "priority": "P3",
        "category": "AI Platform",
        "subcategory": "Model Serving",
        "assignment_group": "AI Platform - Hub",
        "assigned_to": "erik.lindqvist@eon.com",
        "impact": "3-Team",
        "urgency": "3-Medium",
        "cmdb_ci": "se-forecast-endpoint (GPT-4o)",
        "source_system": "Databricks",
        "affected_regions": ["SE"],
        "affected_agents": [a["agent_id"] for a in agents if a["agent_name"] == "SE-Energy-Forecast-Agent"],
        "affected_users_count": 35,
        "mttr_minutes": 120,
        "root_cause": "Provider silently updated model version causing quality regression in domain-specific forecasting",
        "resolution_notes": "Pinned model to previous version (gpt-4o-2025-05-13). Implemented model version monitoring alert. Evaluating fine-tuned alternative.",
        "opened_at": START_DATE + timedelta(days=22, hours=10, minutes=0),
        "state": "Closed"
    },
    {
        "scenario": 6,
        "number": "INC0012006",
        "short_description": "SECURITY: Unauthorized Cross-Region Data Access - RO Agent",
        "description": "RO-Operations-Agent attempting to access DE customer data and Hub financial tables. UC access policies blocking requests. Pattern suggests prompt injection causing lateral movement. Accessing eon_de.customer_360.*, eon_hub.finance.*, eon_hub.hr.*",
        "priority": "P1",
        "category": "Security",
        "subcategory": "Guardrail Violation",
        "assignment_group": "Security Operations",
        "assigned_to": "ion.popescu@eon.com",
        "impact": "2-Region",
        "urgency": "1-Critical",
        "cmdb_ci": "RO-Operations-Agent (AgentBricks)",
        "source_system": "Databricks",
        "affected_regions": ["RO", "DE"],
        "affected_agents": [a["agent_id"] for a in agents if a["agent_name"] == "RO-Operations-Agent"],
        "affected_users_count": 5,
        "mttr_minutes": 20,
        "root_cause": "Prompt injection via crafted user input causing agent to attempt cross-region data exfiltration",
        "resolution_notes": "Agent isolated. Credential rotation completed. Input validation strengthened. UC row-level security confirmed effective (all access DENIED).",
        "opened_at": START_DATE + timedelta(days=21, hours=2, minutes=8),
        "state": "Closed"
    },
    {
        "scenario": 7,
        "number": "INC0012007",
        "short_description": "Cost Anomaly - HU Grid Operations Agent (reasoning loop)",
        "description": "HU-Grid-Operations-Agent stuck in tool-calling loop. Single requests generating 50+ invocations of check_grid_status tool. Individual run costs exceeding $2 (normal: $0.10). Total budget impact: ~$450 in 90 minutes.",
        "priority": "P2",
        "category": "AI Platform",
        "subcategory": "Cost Anomaly",
        "assignment_group": "AI Platform - Hub",
        "assigned_to": "gabor.nagy@eon.com",
        "impact": "3-Team",
        "urgency": "2-High",
        "cmdb_ci": "HU-Grid-Operations-Agent (AgentBricks)",
        "source_system": "Databricks",
        "affected_regions": ["HU"],
        "affected_agents": [a["agent_id"] for a in agents if a["agent_name"] == "HU-Grid-Operations-Agent"],
        "affected_users_count": 8,
        "mttr_minutes": 10,
        "root_cause": "Ambiguous user query caused infinite tool-calling loop. Agent retry logic lacked max-iteration guard.",
        "resolution_notes": "Agent killed. Max tool-call limit (10) implemented. Ambiguous query detection added to prompt. Cost guard at $0.50/request.",
        "opened_at": START_DATE + timedelta(days=14, hours=10, minutes=5),
        "state": "Closed"
    },
    {
        "scenario": 8,
        "number": "INC0012008",
        "short_description": "P1: Multi-Region Agent Failure - Lakebase Tool Degradation",
        "description": "query_energy_prices MCP tool (backed by Lakebase table eon_hub.energy.spot_prices) experiencing 75x latency increase (200ms to 15s). Causing timeout failures across DE, IT, HU, SE agents simultaneously. Connection pool exhaustion detected.",
        "priority": "P1",
        "category": "Data Pipeline",
        "subcategory": "Data Pipeline",
        "assignment_group": "Data Engineering - Hub",
        "assigned_to": "klaus.mueller@eon.com",
        "impact": "1-Enterprise",
        "urgency": "1-Critical",
        "cmdb_ci": "hub-lakebase-energy-prices (Lakebase)",
        "source_system": "New Relic",
        "affected_regions": ["DE", "IT", "HU", "SE"],
        "affected_agents": [a["agent_id"] for a in agents if a["region"] in ["DE", "IT", "HU", "SE"]],
        "affected_users_count": 320,
        "mttr_minutes": 45,
        "root_cause": "Upstream data pipeline delay caused Lakebase connection pool exhaustion under query load",
        "resolution_notes": "Pipeline restarted. Connection pool max increased from 50 to 200. Added circuit breaker pattern to MCP tool. Implemented stale data fallback.",
        "opened_at": START_DATE + timedelta(days=8, hours=6, minutes=12),
        "state": "Closed"
    },
    {
        "scenario": 9,
        "number": "INC0012009",
        "short_description": "COMPLIANCE: Shadow AI Detected - NL Agent Bypassing Gateway",
        "description": "NL-Bedrock-Dev-Agent showing zero AI Gateway traffic for 6 hours while MLflow traces confirm continued operation. Agent calling Amazon Bedrock directly, bypassing governance controls. Cost and access data not captured. Compliance violation.",
        "priority": "P1",
        "category": "Compliance",
        "subcategory": "Shadow AI",
        "assignment_group": "Security Operations",
        "assigned_to": "jan.devries@eon.com",
        "impact": "2-Region",
        "urgency": "1-Critical",
        "cmdb_ci": "NL-Bedrock-Dev-Agent (Bedrock)",
        "source_system": "Databricks",
        "affected_regions": ["NL"],
        "affected_agents": [a["agent_id"] for a in agents if a["agent_name"] == "NL-Bedrock-Dev-Agent"],
        "affected_users_count": 3,
        "mttr_minutes": 180,
        "root_cause": "Developer configured direct Bedrock API access bypassing AI Gateway governance layer",
        "resolution_notes": "Network policy updated to block direct Bedrock API calls. VPC endpoint routing enforced through AI Gateway. Developer counseled. Audit report filed.",
        "opened_at": START_DATE + timedelta(days=14, hours=22, minutes=0),
        "state": "Closed"
    },
    {
        "scenario": 10,
        "number": "INC0012010",
        "short_description": "P1: Cross-Agent Cascading Failure - Hub Orchestrator Retry Storm",
        "description": "DE-Orchestrator-Agent retry storm overwhelming spoke agents. IT-Demand-Forecast-Agent failed (root cause: Azure OpenAI 500). Orchestrator aggressive retry (5x volume) caused rate limiting of HU and RO agents. 80%+ error rate across 4 regions for 30 minutes.",
        "priority": "P1",
        "category": "Infrastructure",
        "subcategory": "AI Gateway",
        "assignment_group": "AI Platform - Hub",
        "assigned_to": "klaus.mueller@eon.com",
        "impact": "1-Enterprise",
        "urgency": "1-Critical",
        "cmdb_ci": "DE-Orchestrator-Agent (AgentBricks)",
        "source_system": "Databricks",
        "affected_regions": ["DE", "IT", "HU", "RO"],
        "affected_agents": [a["agent_id"] for a in agents if a["region"] in ["DE", "IT", "HU", "RO"]],
        "affected_users_count": 450,
        "mttr_minutes": 30,
        "root_cause": "Missing circuit breaker in orchestrator retry logic. Single spoke failure caused cascading overload.",
        "resolution_notes": "Circuit breaker pattern implemented (3 failures = open circuit for 60s). Per-spoke rate limits added. Orchestrator retry backoff changed to exponential. AI Gateway routing updated with spoke-level isolation.",
        "opened_at": START_DATE + timedelta(days=24, hours=9, minutes=17),
        "state": "Closed"
    }
]

# Build incident records
incidents = []
for inc in incident_scenarios:
    opened = inc["opened_at"]
    resolved = opened + timedelta(minutes=inc["mttr_minutes"])
    closed = resolved + timedelta(hours=random.randint(2, 24))
    
    incidents.append({
        "incident_id": str(uuid.uuid5(uuid.NAMESPACE_DNS, inc["number"])),
        "number": inc["number"],
        "short_description": inc["short_description"],
        "description": inc["description"],
        "priority": inc["priority"],
        "state": inc["state"],
        "category": inc["category"],
        "subcategory": inc["subcategory"],
        "assignment_group": inc["assignment_group"],
        "assigned_to": inc["assigned_to"],
        "opened_at": opened,
        "resolved_at": resolved,
        "closed_at": closed,
        "impact": inc["impact"],
        "urgency": inc["urgency"],
        "cmdb_ci": inc["cmdb_ci"],
        "source_alert_id": f"alert-{inc['scenario']:03d}-{str(uuid.uuid4())[:8]}",
        "source_system": inc["source_system"],
        "affected_regions": inc["affected_regions"],
        "affected_agents": inc["affected_agents"],
        "affected_users_count": inc["affected_users_count"],
        "mttr_minutes": inc["mttr_minutes"],
        "root_cause": inc["root_cause"],
        "resolution_notes": inc["resolution_notes"]
    })

# Schema with arrays
incidents_schema = StructType([
    StructField("incident_id", StringType()),
    StructField("number", StringType()),
    StructField("short_description", StringType()),
    StructField("description", StringType()),
    StructField("priority", StringType()),
    StructField("state", StringType()),
    StructField("category", StringType()),
    StructField("subcategory", StringType()),
    StructField("assignment_group", StringType()),
    StructField("assigned_to", StringType()),
    StructField("opened_at", TimestampType()),
    StructField("resolved_at", TimestampType()),
    StructField("closed_at", TimestampType()),
    StructField("impact", StringType()),
    StructField("urgency", StringType()),
    StructField("cmdb_ci", StringType()),
    StructField("source_alert_id", StringType()),
    StructField("source_system", StringType()),
    StructField("affected_regions", ArrayType(StringType())),
    StructField("affected_agents", ArrayType(StringType())),
    StructField("affected_users_count", IntegerType()),
    StructField("mttr_minutes", IntegerType()),
    StructField("root_cause", StringType()),
    StructField("resolution_notes", StringType())
])

incidents_df = spark.createDataFrame(incidents, schema=incidents_schema)
incidents_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.servicenow_incidents")

print(f"✓ ServiceNow Incidents: {len(incidents)} incidents written")
display(incidents_df.select("number", "priority", "short_description", "state", "mttr_minutes", "source_system"))

In [0]:
# ─── ServiceNow Change Requests ─────────────────────────────────────────────
# Change tickets triggered by resolved incidents

change_requests = [
    {
        "number": "CHG0005001",
        "short_description": "Enable multi-region failover for NL AI Gateway endpoints",
        "type": "Normal",
        "state": "Closed",
        "risk": "Medium",
        "parent_incident_id": incidents[0]["incident_id"],
        "requested_by": "jan.devries@eon.com",
        "assignment_group": "AI Platform - Hub",
        "planned_start": incidents[0]["resolved_at"] + timedelta(days=2),
        "planned_end": incidents[0]["resolved_at"] + timedelta(days=2, hours=4),
        "description": "Configure cross-region failover routing for NL endpoints. Primary: NL Azure OpenAI. Failback: DE Azure OpenAI. Auto-switch threshold: 3 consecutive 503s.",
        "cmdb_ci": "nl-gpt4o-endpoint (Azure OpenAI - West Europe)",
        "backout_plan": "Revert AI Gateway routing config to NL-only. Monitoring: check error rates for 1 hour post-deployment."
    },
    {
        "number": "CHG0005002",
        "short_description": "Implement per-region rate limit pools for Monday peak",
        "type": "Normal",
        "state": "Closed",
        "risk": "Low",
        "parent_incident_id": incidents[2]["incident_id"],
        "requested_by": "klaus.mueller@eon.com",
        "assignment_group": "AI Platform - Hub",
        "planned_start": incidents[2]["resolved_at"] + timedelta(days=1),
        "planned_end": incidents[2]["resolved_at"] + timedelta(days=1, hours=2),
        "description": "Increase per-region rate limits by 3x. Implement staggered agent warm-up schedule (DE: 08:45, HU: 08:50, RO: 08:55). Add region-aware token bucket to AI Gateway.",
        "cmdb_ci": "hub-ai-gateway (Shared Multi-Region)",
        "backout_plan": "Revert rate limit configuration to previous values. Disable staggered warm-up."
    },
    {
        "number": "CHG0005003",
        "short_description": "Implement circuit breaker in DE-Orchestrator retry logic",
        "type": "Emergency",
        "state": "Closed",
        "risk": "High",
        "parent_incident_id": incidents[9]["incident_id"],
        "requested_by": "klaus.mueller@eon.com",
        "assignment_group": "AI Platform - Hub",
        "planned_start": incidents[9]["resolved_at"] + timedelta(hours=4),
        "planned_end": incidents[9]["resolved_at"] + timedelta(hours=6),
        "description": "Add circuit breaker pattern: 3 consecutive failures = open circuit for 60s. Exponential backoff on retries. Per-spoke rate limit isolation. AI Gateway routing with spoke-level circuit breaking.",
        "cmdb_ci": "DE-Orchestrator-Agent (AgentBricks)",
        "backout_plan": "Disable circuit breaker, revert to current retry logic. Accept risk of cascade in interim."
    },
    {
        "number": "CHG0005004",
        "short_description": "Block direct Bedrock API access - enforce AI Gateway routing",
        "type": "Emergency",
        "state": "Closed",
        "risk": "Medium",
        "parent_incident_id": incidents[8]["incident_id"],
        "requested_by": "jan.devries@eon.com",
        "assignment_group": "Security Operations",
        "planned_start": incidents[8]["resolved_at"] + timedelta(hours=2),
        "planned_end": incidents[8]["resolved_at"] + timedelta(hours=4),
        "description": "Update VPC/NSG network policies to block direct Amazon Bedrock API calls from compute subnets. All LLM traffic must route through AI Gateway. Add monitoring alert for direct provider API calls.",
        "cmdb_ci": "NL-Bedrock-Dev-Agent (Bedrock)",
        "backout_plan": "Revert network policy. Add temporary exception for NL dev agent while permanent fix is implemented."
    },
    {
        "number": "CHG0005005",
        "short_description": "Add max tool-call limit and cost guard to all agents",
        "type": "Normal",
        "state": "Implement",
        "risk": "Low",
        "parent_incident_id": incidents[6]["incident_id"],
        "requested_by": "gabor.nagy@eon.com",
        "assignment_group": "AI Platform - Hub",
        "planned_start": incidents[6]["resolved_at"] + timedelta(days=3),
        "planned_end": incidents[6]["resolved_at"] + timedelta(days=3, hours=3),
        "description": "Deploy max tool-call limit (10 per request) and cost guard ($0.50/request threshold) across all AgentBricks agents. Add ambiguous query detection to agent prompts.",
        "cmdb_ci": "All AgentBricks Agents (Global)",
        "backout_plan": "Remove tool-call limit and cost guard configurations. Revert agent prompt templates."
    },
    {
        "number": "CHG0005006",
        "short_description": "Increase Lakebase connection pool and add circuit breaker to MCP tool",
        "type": "Normal",
        "state": "Closed",
        "risk": "Medium",
        "parent_incident_id": incidents[7]["incident_id"],
        "requested_by": "klaus.mueller@eon.com",
        "assignment_group": "Data Engineering - Hub",
        "planned_start": incidents[7]["resolved_at"] + timedelta(days=1),
        "planned_end": incidents[7]["resolved_at"] + timedelta(days=1, hours=3),
        "description": "Increase Lakebase connection pool from 50 to 200. Add circuit breaker to query_energy_prices MCP tool (5s timeout, 3 failures = break). Implement stale data fallback returning last known good result.",
        "cmdb_ci": "hub-lakebase-energy-prices (Lakebase)",
        "backout_plan": "Revert connection pool to 50. Remove circuit breaker (MCP tool will timeout naturally). Disable stale data fallback."
    }
]

changes = []
for chg in change_requests:
    changes.append({
        "change_id": str(uuid.uuid5(uuid.NAMESPACE_DNS, chg["number"])),
        "number": chg["number"],
        "short_description": chg["short_description"],
        "type": chg["type"],
        "state": chg["state"],
        "risk": chg["risk"],
        "parent_incident_id": chg["parent_incident_id"],
        "requested_by": chg["requested_by"],
        "assignment_group": chg["assignment_group"],
        "planned_start": chg["planned_start"],
        "planned_end": chg["planned_end"],
        "description": chg["description"],
        "cmdb_ci": chg["cmdb_ci"],
        "backout_plan": chg["backout_plan"]
    })

changes_df = spark.createDataFrame(changes)
changes_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.servicenow_change_requests")

print(f"✓ ServiceNow Change Requests: {len(changes)} change requests written")
display(changes_df.select("number", "type", "short_description", "state", "risk"))

In [0]:
# ─── Summary & Verification ──────────────────────────────────────────────────

tables = [
    "agent_registry", "mlflow_experiments", "mlflow_runs", 
    "mlflow_run_metrics", "ai_gateway_usage", "tool_access_logs",
    "mcp_catalog", "user_profiles",
    "newrelic_infra_metrics", "newrelic_apm_transactions",
    "servicenow_incidents", "servicenow_change_requests"
]

print("="*70)
print()
print("  Run the following queries to verify anomaly injection:")
print("  • SELECT params['error_type'], count(*) FROM mlflow_runs")
print("    WHERE params['error_type'] != '' GROUP BY 1")
print("  • SELECT status_code, count(*) FROM ai_gateway_usage")
print("    WHERE status_code >= 400 GROUP BY 1")
print("  • SELECT count(*) FROM tool_access_logs WHERE granted = false")
print("  • SELECT alert_severity, count(*) FROM newrelic_infra_metrics")
print("    WHERE alert_severity IS NOT NULL GROUP BY 1")
print(f"  AI OBSERVABILITY DEMO DATA — {CATALOG}.{SCHEMA}")
print("="*70)
print()
print("  Run the following queries to verify anomaly injection:")
print("  • SELECT params['error_type'], count(*) FROM mlflow_runs")
print("    WHERE params['error_type'] != '' GROUP BY 1")
print("  • SELECT status_code, count(*) FROM ai_gateway_usage")
print("    WHERE status_code >= 400 GROUP BY 1")
print("  • SELECT count(*) FROM tool_access_logs WHERE granted = false")
print("  • SELECT alert_severity, count(*) FROM newrelic_infra_metrics")
print("    WHERE alert_severity IS NOT NULL GROUP BY 1")
print()

for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.{SCHEMA}.{table}").collect()[0]["cnt"]
    print(f"  {table:<30} {count:>12,} rows")

print()
print("─"*70)
print("  INTEGRATED OBSERVABILITY STACK:")
print("─"*70)
print("  • Databricks: MLflow traces + AI Gateway + UC Access Logs")
print("  • New Relic:  Infra metrics + APM transactions")
print("  • ServiceNow: Incidents + Change Requests")
print()
print("─"*70)
print("  ANOMALY SCENARIOS INJECTED:")
print("─"*70)
print("   1. Provider Outage          (NL, Day 12, 14:00-16:00 CET)")
print("   2. Token Budget Exhaustion   (IT, Days 8-10, gradual)")
print("   3. Rate Limit Cascade        (DE/HU/RO, Days 5/12/19 Mon 9AM)")
print("   4. Guardrail Blocking Surge  (DE HR, Day 18, 10:00-14:00)")
print("   5. Model Drift               (SE, Days 20-27, continuous)")
print("   6. Lateral Movement          (RO, Day 22, 03:00-05:00)")
print("   7. Reasoning Loop            (HU, Day 15, 11:00-12:30)")
print("   8. MCP Tool Degradation      (DE/IT/HU/SE, Day 9, 06:00-06:45)")
print("   9. Shadow AI Gap             (NL, Day 14 20:00 - Day 15 02:00)")
print("  10. Cross-Agent Cascade       (DE→IT→HU→RO, Day 25, 09:15-09:45)")
print("="*70)
print()
print("  Run the following queries to verify anomaly injection:")
print("  • SELECT params['error_type'], count(*) FROM mlflow_runs")
print("    WHERE params['error_type'] != '' GROUP BY 1")
print("  • SELECT status_code, count(*) FROM ai_gateway_usage")
print("    WHERE status_code >= 400 GROUP BY 1")
print("  • SELECT count(*) FROM tool_access_logs WHERE granted = false")
print("  • SELECT alert_severity, count(*) FROM newrelic_infra_metrics")
print("    WHERE alert_severity IS NOT NULL GROUP BY 1")